In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler

# 1. LOAD AND AGGRESSIVE CLEANING
df = pd.read_csv('archive/IPL_dataset.csv')
df.columns = df.columns.str.strip().str.upper()

# Standardize text data (Remove hidden spaces and make uppercase)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip().str.upper()

# 2. NUMERIC CONVERSION & SAFETY
numeric_cols = ['AGE', 'AVG', 'SR', 'HS', '4S', '6S', 'B_AVG', 'B_ECON', 'B_WKTS', 'TAVG', 'TSR', 'B_TAVG', 'B_TECON']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
df.fillna(0, inplace=True)


# We calculate separate strengths so a top batsman can compete with a top bowler
df['BAT_STRENGTH'] = (df['SR'] * 0.6) + (df['AVG'] * 0.4) + (df['6S'] * 2)
df['BOWL_STRENGTH'] = (df['B_WKTS'] * 20) - (df['B_ECON'] * 8)

# Scale both to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))
df[['BAT_STRENGTH', 'BOWL_STRENGTH']] = scaler.fit_transform(df[['BAT_STRENGTH', 'BOWL_STRENGTH']])

# Final impact is the best of their skills
df['ACTUAL_IMPACT'] = df[['BAT_STRENGTH', 'BOWL_STRENGTH']].max(axis=1)

# Train ML Model to learn the 'Impact'
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(df[numeric_cols], df['ACTUAL_IMPACT'])
df['AI_SCORE'] = model.predict(df[numeric_cols])

# 4. TACTICAL 11 SELECTOR
def get_best_playing_11(unavailable_list=[]):
    # Process unavailable list
    unavailable_list = [n.strip().upper() for n in unavailable_list]
    pool = df[~df['PLAYER'].isin(unavailable_list)].copy()
    pool = pool.sort_values(by='AI_SCORE', ascending=False)
    
    final_squad = []

    def pick(query, count, current_pool):
        return current_pool[current_pool['PAYING_ROLE'].str.contains(query, na=False)].head(count)

    # SLOT 1-2: OPENERS (Best Batsmen)
    openers = pick('BATTING|BATSMAN', 2, pool)
    final_squad.append(openers)
    pool = pool[~pool['PLAYER'].isin(openers['PLAYER'])]

    # SLOT 3: WICKETKEEPER
    wk = pick('WK|KEEPER', 1, pool)
    final_squad.append(wk)
    pool = pool[~pool['PLAYER'].isin(wk['PLAYER'])]

    # SLOT 4-5: MIDDLE ORDER (Where Kohli/Rahul/Iyer fit)
    middle = pick('BATTING|BATSMAN', 2, pool)
    final_squad.append(middle)
    pool = pool[~pool['PLAYER'].isin(middle['PLAYER'])]

    # SLOT 6-7: ALL-ROUNDERS (Finishers like Dube/Hardik)
    all_rounders = pick('ALL', 2, pool)
    final_squad.append(all_rounders)
    pool = pool[~pool['PLAYER'].isin(all_rounders['PLAYER'])]

    # SLOT 8-11: BOWLING ATTACK (4 slots)
    bowlers = pick('BOWL', 4, pool)
    final_squad.append(bowlers)

    # Combine and final check
    result_team = pd.concat(final_squad).head(11)
    
    # Fill if any slots are missing due to role unavailability
    if len(result_team) < 11:
        needed = 11 - len(result_team)
        fillers = pool[~pool['PLAYER'].isin(result_team['PLAYER'])].head(needed)
        result_team = pd.concat([result_team, fillers])

    return result_team[['PLAYER', 'TEAM', 'PAYING_ROLE', 'AI_SCORE']].reset_index(drop=True)

# --- EXECUTION ---

# Example: Get the best possible 11 (No exclusions)
print("--- ORIGINAL BEST PLAYING 11 ---")
display(get_best_playing_11(unavailable_list=[]))

# Example: Exclude multiple players
unavailable = ['ROHIT SHARMA', 'MS DHONI','UMESH YADAV','SANDEEP SHARMA','DAVID WARNER','VIRAT KOHLI','DWAYNE BRAVO','SUNIL NARINE']
print(f"\n--- SQUAD EXCLUDING: {unavailable} ---")
display(get_best_playing_11(unavailable_list=unavailable))

--- ORIGINAL BEST PLAYING 11 ---


,PLAYER,TEAM,PAYING_ROLE,AI_SCORE
0,ROHIT SHARMA,MI,BATTING,97.074293
1,MS DHONI,CSK,BATTING,94.652732
2,DAVID WARNER,DC,BATTING,93.726352
3,VIRAT KOHLI,RCB,BATTING,93.703907
4,DWAYNE BRAVO,CSK,ALL ROUNDER,97.221441
5,RAVICHANDRAN ASHWIN,RR,ALL ROUNDER,94.115512
6,YUZVENDRA CHAHAL,RR,BOWLING,94.773182
7,BHUVNESHWAR KUMAR,SRH,BOWLING,93.727412
8,JASPRIT BUMRAH,MI,BOWLING,92.884022
9,UMESH YADAV,KKR,BOWLING,90.947520



--- SQUAD EXCLUDING: ['ROHIT SHARMA', 'MS DHONI', 'UMESH YADAV', 'SANDEEP SHARMA', 'DAVID WARNER', 'VIRAT KOHLI', 'DWAYNE BRAVO', 'SUNIL NARINE'] ---


,PLAYER,TEAM,PAYING_ROLE,AI_SCORE
0,K L RAHUL,LSG,BATTING,73.780143
1,AMBATI RAYUDU,CSK,BATTING,72.610914
2,SANJU SAMSON,RR,BATTING,71.741738
3,SHIKHAR DHAWAN,PK,BATTING,66.771936
4,RAVICHANDRAN ASHWIN,RR,ALL ROUNDER,94.115512
5,RAVINDRA JADEJA,CSK,ALL ROUNDER,90.769728
6,YUZVENDRA CHAHAL,RR,BOWLING,94.773182
7,BHUVNESHWAR KUMAR,SRH,BOWLING,93.727412
8,JASPRIT BUMRAH,MI,BOWLING,92.884022
9,KAGISO RABADA,PK,BOWLING,82.778247
